# 04 - Optimización de Modelos para Edge/Mobile

Un modelo que funciona en un servidor GPU no sirve directamente en un móvil o Raspberry Pi.
Necesitamos **optimizarlo**: reducir tamaño y latencia manteniendo la accuracy.

## Contenido
1. El problema: por qué optimizar
2. Quantización: de float32 a int8
3. Exportación ONNX
4. Conversión a TFLite
5. Compilación para Edge TPU (Google Coral)
6. Benchmark: comparar todos los formatos
7. Pipeline completo con MLForge

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import numpy as np
import os
import time

device = torch.device('cpu')  # Usamos CPU para benchmarks justos

## 1. El Problema: Restricciones de Edge/Mobile

| Restricción | Servidor | Raspberry Pi 4 | Móvil | Coral Edge TPU |
|-------------|---------|-----------------|-------|----------------|
| RAM | 64+ GB | 2-8 GB | 4-8 GB | 8 MB on-chip |
| Compute | GPU (TFLOPS) | CPU 1.5GHz | NPU/GPU móvil | 4 TOPS INT8 |
| Storage | TB | 32 GB SD | 64-256 GB | — |
| Batería | Ilimitada | 5V USB | Limitada | USB powered |
| Modelo máx | ~10 GB | ~100 MB | ~50 MB | ~8 MB |

**Un MobileNetV3 en float32**: ~10 MB
**Quantizado a INT8**: ~2.5 MB (¡4x más pequeño!)
**Latencia en Coral**: ~3ms (vs ~100ms en CPU de RPi)

In [ ]:
# Modelo de referencia: MobileNetV3 Small
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.eval()

# Tamaño del modelo en memoria
param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
total_size_mb = (param_size + buffer_size) / 1024 / 1024

print(f"MobileNetV3 Small:")
print(f"  Parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Tamaño en memoria: {total_size_mb:.2f} MB")
print(f"  Precisión: float32 (cada peso = 4 bytes)")

# Benchmark de latencia en CPU
dummy_input = torch.randn(1, 3, 224, 224)
times = []
with torch.no_grad():
    for _ in range(50):
        start = time.perf_counter()
        model(dummy_input)
        times.append((time.perf_counter() - start) * 1000)

print(f"  Latencia CPU: {np.median(times):.1f} ms (mediana)")

## 2. Quantización: Reducir Precisión

La idea: usar **int8** (1 byte) en lugar de **float32** (4 bytes).

Tipos de quantización:

| Tipo | Calibración | Reducción | Pérdida accuracy |
|------|------------|-----------|------------------|
| Dynamic | No | ~4x tamaño | Mínima (<1%) |
| Static (PTQ) | Sí (datos) | ~4x tamaño + ~2x velocidad | Baja (~1-2%) |
| QAT | Sí (entrena) | ~4x tamaño + ~2x velocidad | Muy baja (<0.5%) |

In [ ]:
# Quantización dinámica (la más simple, sin calibración)
model_dynamic = torch.ao.quantization.quantize_dynamic(
    model,
    {nn.Linear},  # Quantizar solo capas lineales
    dtype=torch.qint8,
)

# Guardar y comparar tamaños
os.makedirs('output', exist_ok=True)

torch.save(model.state_dict(), 'output/model_fp32.pth')
torch.save(model_dynamic.state_dict(), 'output/model_dynamic_int8.pth')

size_fp32 = os.path.getsize('output/model_fp32.pth') / 1024 / 1024
size_int8 = os.path.getsize('output/model_dynamic_int8.pth') / 1024 / 1024

print(f"FP32:         {size_fp32:.2f} MB")
print(f"Dynamic INT8: {size_int8:.2f} MB")
print(f"Reducción:    {size_fp32/size_int8:.1f}x")

# Benchmark quantizado
times_q = []
with torch.no_grad():
    for _ in range(50):
        start = time.perf_counter()
        model_dynamic(dummy_input)
        times_q.append((time.perf_counter() - start) * 1000)

print(f"\nLatencia FP32: {np.median(times):.1f} ms")
print(f"Latencia INT8: {np.median(times_q):.1f} ms")
print(f"Speedup:       {np.median(times)/np.median(times_q):.2f}x")

## 3. Exportación a ONNX

ONNX (Open Neural Network Exchange) es un formato intermedio universal.
Desde ONNX puedes convertir a casi cualquier formato de despliegue.

```
PyTorch → ONNX → TFLite
                → CoreML
                → TensorRT
                → ONNX Runtime (CPU/GPU/mobile)
```

In [ ]:
import onnx
import onnxruntime as ort

# Exportar a ONNX
onnx_path = 'output/model.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    opset_version=17,
    input_names=['image'],
    output_names=['predictions'],
    dynamic_axes={'image': {0: 'batch'}, 'predictions': {0: 'batch'}},
)

# Verificar
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
size_onnx = os.path.getsize(onnx_path) / 1024 / 1024
print(f"ONNX exportado: {size_onnx:.2f} MB")

# Benchmark ONNX Runtime
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name

times_onnx = []
for _ in range(50):
    start = time.perf_counter()
    session.run(None, {input_name: dummy_input.numpy()})
    times_onnx.append((time.perf_counter() - start) * 1000)

print(f"ONNX Runtime latencia: {np.median(times_onnx):.1f} ms")

# Validar: comparar outputs
with torch.no_grad():
    pytorch_out = model(dummy_input).numpy()
onnx_out = session.run(None, {input_name: dummy_input.numpy()})[0]
max_diff = np.max(np.abs(pytorch_out - onnx_out))
print(f"Max diferencia PyTorch vs ONNX: {max_diff:.6f} (debe ser < 0.001)")

## 4. Conversión a TFLite

TFLite es el formato estándar para Android y Raspberry Pi.
Soporta quantización nativa con calibración.

In [ ]:
# Conversión ONNX → TFLite (via onnx2tf o onnx-tf)
try:
    import onnx2tf
    
    # Convertir ONNX → SavedModel → TFLite
    onnx2tf.convert(
        input_onnx_file_path=onnx_path,
        output_folder_path='output/saved_model',
        non_verbose=True,
    )
    
    import tensorflow as tf
    
    # Float32 TFLite
    converter = tf.lite.TFLiteConverter.from_saved_model('output/saved_model')
    tflite_fp32 = converter.convert()
    
    with open('output/model_fp32.tflite', 'wb') as f:
        f.write(tflite_fp32)
    
    # Float16 quantized TFLite
    converter = tf.lite.TFLiteConverter.from_saved_model('output/saved_model')
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_fp16 = converter.convert()
    
    with open('output/model_fp16.tflite', 'wb') as f:
        f.write(tflite_fp16)
    
    # INT8 quantized (full integer)
    converter = tf.lite.TFLiteConverter.from_saved_model('output/saved_model')
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # Representative dataset para calibración
    def representative_dataset():
        for _ in range(100):
            data = np.random.rand(1, 224, 224, 3).astype(np.float32)
            yield [data]
    
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_int8 = converter.convert()
    
    with open('output/model_int8.tflite', 'wb') as f:
        f.write(tflite_int8)
    
    # Comparar tamaños
    for name, data in [('FP32', tflite_fp32), ('FP16', tflite_fp16), ('INT8', tflite_int8)]:
        print(f"TFLite {name}: {len(data)/1024/1024:.2f} MB")

except ImportError as e:
    print(f"Nota: {e}")
    print("Instala onnx2tf: pip install onnx2tf")
    print("O usa MLForge: mlforge export --config config.yaml --formats tflite")

## 5. Google Coral Edge TPU

El Edge TPU solo ejecuta modelos **full INT8 quantized** en formato TFLite.
Necesitamos un paso extra de compilación:

```bash
# Instalar el compilador Edge TPU
# https://coral.ai/docs/edgetpu/compiler/
edgetpu_compiler model_int8.tflite
```

El compilador particiona las operaciones:
- **On-chip**: operaciones soportadas por el TPU (~4 TOPS INT8)
- **On-CPU**: operaciones no soportadas (fallback lento)

Un buen modelo para Coral debe tener >95% de operaciones en TPU.

### Modelos compatibles con Edge TPU:
- MobileNetV1/V2/V3 (diseñados para esto)
- EfficientNet-Lite
- SSD MobileNet (detección)
- No: modelos con operaciones dinámicas o no cuantizables

In [ ]:
# Simulación de lo que hace el Edge TPU compiler
import shutil

has_edgetpu_compiler = shutil.which('edgetpu_compiler') is not None

if has_edgetpu_compiler:
    import subprocess
    result = subprocess.run(
        ['edgetpu_compiler', '-s', '-o', 'output', 'output/model_int8.tflite'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode == 0:
        size = os.path.getsize('output/model_int8_edgetpu.tflite') / 1024 / 1024
        print(f"Edge TPU model: {size:.2f} MB")
else:
    print("Edge TPU compiler no instalado.")
    print("En producción, usa: mlforge export --formats edgetpu")
    print("\nEl compilador se instala con:")
    print("  curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -")
    print("  echo 'deb https://packages.cloud.google.com/apt coral-edgetpu-stable main' | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list")
    print("  sudo apt-get update && sudo apt-get install edgetpu-compiler")

## 6. Tabla Resumen de Formatos

| Formato | Tamaño | Latencia | Plataforma |
|---------|--------|----------|------------|
| PyTorch (.pth) | ~10 MB | ~50ms CPU | Servidor, investigación |
| ONNX (.onnx) | ~10 MB | ~30ms CPU | Universal, web, mobile |
| TFLite FP32 | ~10 MB | ~40ms CPU | Android, RPi |
| TFLite FP16 | ~5 MB | ~35ms CPU | Android, RPi |
| TFLite INT8 | ~2.5 MB | ~15ms CPU | Android, RPi, Edge TPU |
| Edge TPU | ~2.5 MB | ~3ms TPU | Google Coral |
| CoreML | ~10 MB | ~5ms NPU | iPhone, iPad, Mac |
| TF.js | ~10 MB | ~50ms GPU | Navegadores web |

## 7. Pipeline Completo con MLForge

Todo lo anterior en 3 comandos:

```bash
# 1. Entrenar
mlforge train --config config.yaml

# 2. Exportar a TODOS los formatos
mlforge export --config config.yaml --formats all

# 3. Benchmark comparativo
mlforge benchmark --model-dir exported_models/ --config config.yaml --runs 100
```

En `config.yaml`:
```yaml
export:
  output_dir: exported_models
  formats:
    - onnx: {}
    - tflite: {quantize: int8}
    - coreml: {quantize: float16}
    - tfjs: {}
    - edgetpu: {}
```

**Siguiente notebook:** Deploy Everywhere — cómo llevar tu modelo a Android, iOS, web y Raspberry Pi.